In [1]:
print("Hello World")

Hello World


In [2]:
# Loading in a dataset
import seaborn as sns
df = sns.load_dataset("titanic")
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [3]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import LabelEncoder

# 1. Check for missing values and handle them.
# 2. Check for duplicate rows and remove them.
# 3. Inspect the "Sex" column for inconsistencies and standardize values.
# 4. Detect and handle outliers in the "Age" and "Fare" columns.
# 5. Create a new "Family_Size" feature based on "SibSp" and "Parch".
# 6. Convert the "Embarked" column to numerical values using label encoding.
# 7. If a "Ticket_Purchase_Date" column existed, convert it to datetime and extract the year and month.
# 8. Clean the "Name" column by removing titles and converting to lowercase.
# 9. Bin the "Age" column into categories like 'Child', 'Teen', 'Adult', and 'Senior'.

In [5]:
# Load the Titanic dataset
titanic_df = sns.load_dataset("titanic")

In [26]:
#titanic_df['age'] = titanic_df['age'].fillna(0)
#titanic_df['embarked'] = titanic_df['embarked'].fillna('Z')
#titanic_df['embark_town'] = titanic_df['embark_town'].fillna('Nowhere')
titanic_df['deck'] = titanic_df['deck'].fillna('A')

In [22]:
print(titanic_df[['embark_town', 'deck']])

     embark_town deck
0    Southampton  NaN
1      Cherbourg    C
2    Southampton  NaN
3    Southampton    C
4    Southampton  NaN
..           ...  ...
886  Southampton  NaN
887  Southampton    B
888  Southampton  NaN
889    Cherbourg    C
890   Queenstown  NaN

[891 rows x 2 columns]


In [27]:
# 1. Check for missing values and handle them.
# Solution:



missing_values = titanic_df.isnull().sum()
print("Missing values before cleaning:")
print(missing_values)

Missing values before cleaning:
survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
deck           0
embark_town    0
alive          0
alone          0
dtype: int64


In [65]:
# Handle missing values
titanic_df['age'].fillna(titanic_df['age'].median(), inplace=True)
titanic_df.dropna(subset=['embarked'], inplace=True)

# Verify changes
print("\nMissing values after cleaning:")
print(titanic_df.isnull().sum())


Missing values after cleaning:
survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
deck           0
embark_town    0
alive          0
alone          0
family_size    0
dtype: int64


In [32]:
#print(titanic_df['sex'])

#filtered = titanic_df[~titanic_df['sex'].isin(['m', 'f'])]

#print(filtered)

titanic_df['sex'].unique()

array(['male', 'female'], dtype=object)

In [36]:
# 2. Check for duplicate rows and remove them.
# Solution:
duplicates = titanic_df.duplicated().sum()
print(f"\nNumber of duplicate rows before cleaning: {duplicates}")


Number of duplicate rows before cleaning: 107


In [37]:
# Remove duplicates
titanic_df.drop_duplicates(inplace=True)

In [38]:
# Verify
duplicates_after = titanic_df.duplicated().sum()
print(f"Number of duplicate rows after cleaning: {duplicates_after}")

Number of duplicate rows after cleaning: 0


In [40]:
# 3. Inspect the "Sex" column for inconsistencies and standardize values.
# Solution:
print("\nUnique values in 'Sex' column before cleaning:")
print(titanic_df['sex'].unique())


Unique values in 'Sex' column before cleaning:
['male' 'female']


In [42]:
# Standardize values (e.g., convert 'male' and 'female' to lowercase)
titanic_df['sex'] = titanic_df['sex'].str.lower()

In [44]:
# Verify
print("\nUnique values in 'Sex' column after cleaning:")
print(titanic_df['sex'].unique())


Unique values in 'Sex' column after cleaning:
['male' 'female']


In [47]:
# 4. Detect and handle outliers in the "Age" and "Fare" columns.
# Solution:
Q1_age = titanic_df['age'].quantile(0.25)
Q3_age = titanic_df['age'].quantile(0.75)
IQR_age = Q3_age - Q1_age

Q1_fare = titanic_df['fare'].quantile(0.25)
Q3_fare = titanic_df['fare'].quantile(0.75)
IQR_fare = Q3_fare - Q1_fare

In [49]:
# Detect outliers in 'Age' and 'Fare' columns
age_outliers = titanic_df[(titanic_df['age'] < (Q1_age - 1.5 * IQR_age)) | (titanic_df['age'] > (Q3_age + 1.5 * IQR_age))]
fare_outliers = titanic_df[(titanic_df['fare'] < (Q1_fare - 1.5 * IQR_fare)) | (titanic_df['fare'] > (Q3_fare + 1.5 * IQR_fare))]

In [51]:
# Print outliers
print("\nAge outliers:")
print(age_outliers[['age', 'fare']])

print("\nFare outliers:")
print(fare_outliers[['age', 'fare']])


Age outliers:
      age     fare
96   71.0  34.6542
116  70.5   7.7500
493  71.0  49.5042
630  80.0  30.0000
672  70.0  10.5000
745  70.0  71.0000
851  74.0   7.7750

Fare outliers:
      age      fare
27   19.0  263.0000
31    0.0  146.5208
34   28.0   82.1708
52   49.0   76.7292
61   38.0   80.0000
..    ...       ...
829  62.0   80.0000
835  39.0   83.1583
849   0.0   89.1042
856  45.0  164.8667
879  56.0   83.1583

[102 rows x 2 columns]


In [52]:
# Remove outliers (optional)
titanic_df = titanic_df[~((titanic_df['age'] < (Q1_age - 1.5 * IQR_age)) | (titanic_df['age'] > (Q3_age + 1.5 * IQR_age)))]
titanic_df = titanic_df[~((titanic_df['fare'] < (Q1_fare - 1.5 * IQR_fare)) | (titanic_df['fare'] > (Q3_fare + 1.5 * IQR_fare)))]

In [53]:
# Verify changes
print(f"\nRows after removing outliers: {titanic_df.shape[0]}")


Rows after removing outliers: 675


In [57]:
# 5. Create a new "Family_Size" feature based on "SibSp" and "Parch".
# Solution:
titanic_df['family_size'] = titanic_df['sibsp'] + titanic_df['parch'] + 1

In [59]:
# Verify by displaying the new column
print("\nFamily Size feature:")
print(titanic_df[['sibsp', 'parch', 'family_size']].head())


Family Size feature:
   sibsp  parch  family_size
0      1      0            2
1      1      0            2
2      0      0            1
3      1      0            2
4      0      0            1


In [60]:
# 6. Convert the "Embarked" column to numerical values using label encoding.
# Solution:
label_encoder = LabelEncoder()


In [62]:
# Encode 'Embarked' column
titanic_df['embarked'] = label_encoder.fit_transform(titanic_df['embarked'])

In [63]:
# Verify
print("\n'Embarked' column after label encoding:")
print(titanic_df[['embarked']].head())


'Embarked' column after label encoding:
   embarked
0         2
1         0
2         2
3         2
4         2


In [67]:
# 7. If a "Ticket_Purchase_Date" column existed, convert it to datetime and extract the year and month.
# Solution:
# Example: Add a 'Ticket_Purchase_Date' column for demonstration
#titanic_df['ticket_purchase_date'] = ['2021-01-10', '2021-05-23', '2021-07-15', '2021-09-09', '2021-11-18']

dates = ['2021-01-10', '2021-05-23', '2021-07-15', '2021-09-09', '2021-11-18']

titanic_df['ticket_purchase_date'] = np.resize(dates, len(titanic_df))

In [69]:
# Convert to datetime
titanic_df['ticket_purchase_date'] = pd.to_datetime(titanic_df['ticket_purchase_date'])

In [70]:
# Extract year and month
titanic_df['purchase_year'] = titanic_df['ticket_purchase_date'].dt.year
titanic_df['purchase_month'] = titanic_df['ticket_purchase_date'].dt.month

In [71]:
# Verify
print("\nTicket purchase date and extracted year/month:")
print(titanic_df[['ticket_purchase_date', 'purchase_year', 'purchase_month']].head())


Ticket purchase date and extracted year/month:
  ticket_purchase_date  purchase_year  purchase_month
0           2021-01-10           2021               1
1           2021-05-23           2021               5
2           2021-07-15           2021               7
3           2021-09-09           2021               9
4           2021-11-18           2021              11


In [79]:
print(titanic_df.columns)

Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town',
       'alive', 'alone', 'family_size', 'ticket_purchase_date',
       'purchase_year', 'purchase_month'],
      dtype='object')


In [81]:
# 8. Clean the "Name" column by removing titles and converting to lowercase.
# Solution:
titanic_df['name'] = titanic_df['name'].apply(lambda x: re.sub(r'(Mr\.|Mrs\.|Dr\.)', '', x))
titanic_df['name'] = titanic_df['name'].str.lower()

titanic_df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,family_size,ticket_purchase_date,purchase_year,purchase_month,name
0,0,3,male,22.0,1,0,7.2500,2,Third,man,True,A,Southampton,no,False,2,2021-01-10,2021,1,aa
1,1,1,female,38.0,1,0,71.2833,0,First,woman,False,C,Cherbourg,yes,False,2,2021-05-23,2021,5,aa
2,1,3,female,26.0,0,0,7.9250,2,Third,woman,False,A,Southampton,yes,True,1,2021-07-15,2021,7,aa
3,1,1,female,35.0,1,0,53.1000,2,First,woman,False,C,Southampton,yes,False,2,2021-09-09,2021,9,aa
4,0,3,male,35.0,0,0,8.0500,2,Third,man,True,A,Southampton,no,True,1,2021-11-18,2021,11,aa


In [83]:
# Verify
print("\nName column after normalization:")
print(titanic_df[['name']].head())


Name column after normalization:
  name
0   aa
1   aa
2   aa
3   aa
4   aa


In [84]:
# 9. Bin the "Age" column into categories like 'Child', 'Teen', 'Adult', and 'Senior'.
# Solution:
bins = [0, 12, 18, 64, 100]
labels = ['Child', 'Teen', 'Adult', 'Senior']

In [86]:
# Create new 'Age_Group' feature
titanic_df['age_group'] = pd.cut(titanic_df['age'], bins=bins, labels=labels)

In [88]:
# Verify
print("\nAge and corresponding Age Group:")
print(titanic_df[['age', 'age_group']].head())


Age and corresponding Age Group:
    age age_group
0  22.0     Adult
1  38.0     Adult
2  26.0     Adult
3  35.0     Adult
4  35.0     Adult


In [89]:
# Loading in a dataset
import seaborn as sns
dfv2 = sns.load_dataset("titanic")
dfv2.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [91]:
# Functions
def sayHello(df):
    df.drop_duplicates(inplace=True)
    dfOutput = df.dropna()

    return dfOutput

In [92]:
dfv3 = sayHello(dfv2)
dfv3.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
6,0,1,male,54.0,0,0,51.8625,S,First,man,True,E,Southampton,no,True
10,1,3,female,4.0,1,1,16.7000,S,Third,child,False,G,Southampton,yes,False
11,1,1,female,58.0,0,0,26.5500,S,First,woman,False,C,Southampton,yes,True


In [93]:
def DFSeperate(dfMaster):
    df_numeric = dfMaster.select_dtypes(include='number')

    clean_mask = pd.Series(True, index=dfMaster.index)

    for col in df_numeric.columns:
        Q1 = dfMaster[col].quantile(0.25)
        Q3 = dfMaster[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        col_mask = dfMaster[col].between(lower, upper)

        clean_mask &= col_mask

    clean_df = dfMaster[clean_mask]
    outlier_df = dfMaster[~clean_mask]

    return clean_df, outlier_df


In [96]:
clean_df, outlier_df = DFSeperate(dfv3)

clean_df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
6,0,1,male,54.0,0,0,51.8625,S,First,man,True,E,Southampton,no,True
11,1,1,female,58.0,0,0,26.5500,S,First,woman,False,C,Southampton,yes,True
23,1,1,male,28.0,0,0,35.5000,S,First,man,True,A,Southampton,yes,True


In [95]:
outlier_df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
10,1,3,female,4.0,1,1,16.70,S,Third,child,False,G,Southampton,yes,False
21,1,2,male,34.0,0,0,13.00,S,Second,man,True,D,Southampton,yes,True
27,0,1,male,19.0,3,2,263.00,S,First,man,True,C,Southampton,no,False
66,1,2,female,29.0,0,0,10.50,S,Second,woman,False,F,Southampton,yes,True
75,0,3,male,25.0,0,0,7.65,S,Third,man,True,F,Southampton,no,True
